In [ ]:
!pip install -U pandas==2.2.2 numpy==2.0.2 scipy==1.16.3

In [ ]:
import numpy as np
from scipy import stats
import pandas as pd

# Simulate reference and new data
np.random.seed(42)
n_features = 5
n_samples = 1000

# Reference (training) data: assume normal distributions
ref_data = np.random.normal(loc=50, scale=5, size=(n_samples, n_features))

# Today's data: introduce mild drifts for some features
today_data = np.copy(ref_data)
today_data[:, 0] += np.random.normal(3, 1, n_samples)     # mean shift
today_data[:, 1] *= np.random.normal(1.2, 0.05, n_samples) # scale shift
today_data[:, 2] += np.random.normal(0, 0.1, n_samples)    # small noise
# Features 3 and 4 remain roughly stable

# Convert to DataFrame
feature_names = [f"feature_{i+1}" for i in range(n_features)]
ref_df = pd.DataFrame(ref_data, columns=feature_names)
today_df = pd.DataFrame(today_data, columns=feature_names)

# Thresholds
mean_threshold = 2.0
std_threshold = 2.0
alpha = 0.01  # significance level for KS test

# Drift detection per feature
def detect_drift(ref_df, today_df):
    results = []
    for col in ref_df.columns:
        ref_values = ref_df[col].values
        today_values = today_df[col].values

        mean_diff = abs(today_values.mean() - ref_values.mean())
        std_diff = abs(today_values.std() - ref_values.std())
        ks_stat, p_value = stats.ks_2samp(today_values, ref_values, method='asymp')

        drift_flag = (
            (mean_diff > mean_threshold)
            or (std_diff > std_threshold)
            or (p_value < alpha)
        )

        results.append({
            "Feature": col,
            "Ref_Mean": ref_values.mean(),
            "Today_Mean": today_values.mean(),
            "Mean_Diff": mean_diff,
            "Ref_Std": ref_values.std(),
            "Today_Std": today_values.std(),
            "Std_Diff": std_diff,
            "KS_Stat": ks_stat,
            "p_value": p_value,
            "Drift": drift_flag
        })
    return pd.DataFrame(results)

# Run detection and display
report = detect_drift(ref_df, today_df)
print("\nFeature Drift Report\n")
print(report.to_string(index=False))

# Aggregate summary
drifted = report[report["Drift"] == True]
print("\nSummary:")
print(f"Total features checked: {len(report)}")
print(f"Features with detected drift: {len(drifted)}")

if not drifted.empty:
    print("\nDrift detected in:")
    for f in drifted["Feature"]:
        print(f" - {f}")
else:
    print("No drift detected.")